In [ ]:
## SESSION 1 PASQUALE BIFULCO

In [28]:
!pip install contexttimer 

In [12]:

## EXERCISE 3 ##

from contexttimer import Timer
import functools

def timer_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        with Timer() as t:
            result = func(*args, **kwargs)
        print(f"{func.__name__} took {t.elapsed:.4f} seconds")
        return result
    return wrapper

# Test del decorator
@timer_decorator
def slow_function():
    total = 0
    for i in range(1000000):
        total += i
    return total

result = slow_function()
print(f"Result: {result}")

#contexttimer measure the execution time of the code! 


slow_function took 0.0554 seconds
Result: 499999500000


In [13]:
## EXERCISE 4 ## 

import numpy as np
import pandas as pd
from contexttimer import Timer

# Create Dataframe with MultiIndex
ndates = 3000 # number of dates 
dates = pd.bdate_range(end=pd.Timestamp.now(), periods=ndates)
nsyms = 500 # number of ticker ---> 3000 * 500 = 1.500.000
symbols = [hex(n) for n in range(nsyms)]

index = pd.MultiIndex.from_product([dates, symbols], names=['date', 'symbol'])
df = pd.DataFrame(np.random.randn(ndates * nsyms), index=index, columns=['price'])

print(f"Shape: {df.shape}")
print(df.tail(5))

## dataframe with 1.5M of rows and just 1 columns given by the price of the symbol! ## 


Shape: (1500000, 1)
                      price
date       symbol          
2026-06-10 0x1ef   1.668274
           0x1f0   2.179286
           0x1f1   0.687284
           0x1f2   0.852875
           0x1f3  -1.354473


In [31]:
## DELETE 10 % OF DATA AND TRYING TO SIMULATE MISSING VALUES ----> FORWARD FILLING
p_drop = 0.1
keep = np.random.rand(ndates * nsyms) > p_drop
df_mv = df.loc[keep].copy()
print(f"Original Rows: {len(df)}")
print(f"Rows after drop: {len(df_mv)}")

# Method 1  - groupby ffill
print("\nMethod 1: groupby ffill")
with Timer() as t:
    df_mv.reindex(index).groupby('symbol').ffill()
print(f"Tempo: {t.elapsed:.4f} seconds")

# Method 2 - unstack ffill stack
print("\nMethod 2: unstack → ffill → stack")
with Timer() as t:
    df_mv.unstack().ffill().stack()
print(f"Tempo: {t.elapsed:.4f} seconds")

## groupyby ffill ---> 0.06560 secc > unstack + ffill ( vectorization )  = 0.4374 sec

Original Rows: 1500000
Rows after drop: 1349221

Method 1: groupby ffill
Tempo: 0.5462 seconds

Method 2: unstack → ffill → stack
Tempo: 0.3197 seconds


C:\Users\pasqu\AppData\Local\Temp\ipykernel_19648\827138979.py:17: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_mv.unstack().ffill().stack()


In [32]:
import pyarrow as pa
import os

# Method 1 - Store as CSV
print("=== CSV ===")
with Timer() as t:
    df.to_csv('temp.csv')
print(f"Write: {t.elapsed:.4f} seconds")

with Timer() as t:
    df_csv = pd.read_csv('temp.csv', index_col=['date', 'symbol'])
print(f"Read:  {t.elapsed:.4f} seconds")
print(f"Size:  {os.path.getsize('temp.csv') / 1e6:.1f} MB")

# Metodo 2 - Store as Parquet
print("\n=== PARQUET ===")
with Timer() as t:
    df.to_parquet('temp.parquet')
print(f"Write: {t.elapsed:.4f} seconds")

with Timer() as t:
    df_parquet = pd.read_parquet('temp.parquet')
print(f"Read:  {t.elapsed:.4f} seconds")
print(f"Size:  {os.path.getsize('temp.parquet') / 1e6:.1f} MB")

# Metodo 3 - Store as Feather
print("\n=== FEATHER ===")
with Timer() as t:
    df.to_feather('temp.feather')
print(f"Write: {t.elapsed:.4f} seconds")

with Timer() as t:
    df_feather = pd.read_feather('temp.feather')
print(f"Read:  {t.elapsed:.4f} seconds")
print(f"Size:  {os.path.getsize('temp.feather') / 1e6:.1f} MB")


## CSV is the slowest and largest, avoid for big datasets
# Parquet offer the best compression ( 4x smaller than CSV ) -- ideal for long term storage 
# Feather is the fastest for write and read ---- ideal for daily research workflow

=== CSV ===
Write: 3.5693 seconds
Read:  1.2201 seconds
Size:  55.6 MB

=== PARQUET ===
Write: 0.4215 seconds
Read:  0.2794 seconds
Size:  12.7 MB

=== FEATHER ===
Write: 0.1855 seconds
Read:  0.2847 seconds
Size:  18.2 MB


In [33]:
## EXERCISE 5 : SPLIT THE DATASET IN A MULTIPLE FILE INSTEAD OF A SINGLE FILE!

import os

# Sharding by date
os.makedirs('data_by_date', exist_ok=True)

print("=== SHARDING BY DATE ===")
with Timer() as t:
    for date, group in df.groupby('date'):
        filename = f"data_by_date/{str(date)[:10]}.parquet"
        group.to_parquet(filename)
print(f"Write time: {t.elapsed:.4f} seconds")
print(f"Files created: {len(os.listdir('data_by_date'))}")

# Sharding by symbol
os.makedirs('data_by_symbol', exist_ok=True)

print("\n=== SHARDING BY SYMBOL ===")
with Timer() as t:
    for symbol, group in df.groupby('symbol'):
        filename = f"data_by_symbol/{symbol}.parquet"
        group.to_parquet(filename)
print(f"Write time: {t.elapsed:.4f} seconds")
print(f"Files created: {len(os.listdir('data_by_symbol'))}")

=== SHARDING BY DATE ===
Write time: 14.6142 seconds
Files created: 3000

=== SHARDING BY SYMBOL ===
Write time: 3.2903 seconds
Files created: 500


In [34]:
## EXERCISE 6 

## PARALLELIZATION 

import multiprocessing
from concurrent.futures import ThreadPoolExecutor

# Function that compute the average
def compute_daily_average(date):
    return df.xs(date, level='date')['price'].mean()

# List of all dates 
all_dates = df.index.get_level_values('date').unique().tolist()
print(f"Number of Dates: {len(all_dates)}")

# Method 1 : Simple Loop
print("\n=== Normal Loop ===")
with Timer() as t:
    results_loop = [compute_daily_average(d) for d in all_dates]
print(f"Time: {t.elapsed:.4f} seconds")

## 1.0703 sec is our baseline to compute simple average with a simple loop

Number of Dates: 3000

=== Normal Loop ===
Time: 1.0137 seconds


In [35]:
# Method 2 - Multithreading
print("=== MULTITHREADING ===")
with Timer() as t:
    with ThreadPoolExecutor() as executor:
        results_thread = list(executor.map(compute_daily_average, all_dates))
print(f"Time: {t.elapsed:.4f} seconds")

=== MULTITHREADING ===
Time: 1.5253 seconds


In [36]:
from joblib import Parallel, delayed

print("=== MULTIPROCESSING (joblib) ===")
with Timer() as t:
    results_mp = Parallel(n_jobs=-1)(
        delayed(compute_daily_average)(d) for d in all_dates
    )
print(f"Time: {t.elapsed:.4f} seconds")

# for this type of task Parallelization is not so convenient since the time of execution is greater than the others. 

=== MULTIPROCESSING (joblib) ===
Time: 2.4224 seconds


In [25]:
!pip install apscheduler


   -------------------- ------------------- 1/2 [apscheduler]
   ---------------------------------------- 2/2 [apscheduler]



In [26]:
##EXERCISE 7 

from apscheduler.schedulers.blocking import BlockingScheduler
from datetime import datetime

def job1_download():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Job 1: Downloading data...")

def job2_features():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Job 2: Computing features...")

def job3_forecast():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Job 3: Generating forecasts...")

def job4_portfolio():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Job 4: Optimizing portfolio...")

scheduler = BlockingScheduler()
scheduler.add_job(job1_download, 'interval', seconds=2)
scheduler.add_job(job2_features, 'interval', seconds=4)
scheduler.add_job(job3_forecast, 'interval', seconds=6)
scheduler.add_job(job4_portfolio, 'interval', seconds=8)

print("Starting scheduler - press Ctrl+C to stop")
scheduler.start()

# apscheduler can be used to reproduce at regular interval the execution of the function. Here we have created 4 jobs and after a given interval the process will be repeated. 

Starting scheduler - press Ctrl+C to stop
[20:06:33] Job 1: Downloading data...
[20:06:35] Job 1: Downloading data...
[20:06:35] Job 2: Computing features...
[20:06:37] Job 1: Downloading data...
[20:06:37] Job 3: Generating forecasts...
[20:06:39] Job 1: Downloading data...[20:06:39] Job 2: Computing features...

[20:06:39] Job 4: Optimizing portfolio...
[20:06:41] Job 1: Downloading data...
[20:06:43] Job 1: Downloading data...[20:06:43] Job 2: Computing features...
[20:06:43] Job 3: Generating forecasts...

[20:06:45] Job 1: Downloading data...
[20:06:47] Job 1: Downloading data...[20:06:47] Job 2: Computing features...
[20:06:47] Job 4: Optimizing portfolio...

[20:06:49] Job 1: Downloading data...[20:06:49] Job 3: Generating forecasts...

[20:06:51] Job 1: Downloading data...[20:06:51] Job 2: Computing features...

[20:06:53] Job 1: Downloading data...
[20:06:55] Job 1: Downloading data...[20:06:55] Job 2: Computing features...
[20:06:55] Job 3: Generating forecasts...

[20:06:55]

KeyboardInterrupt: 